# Tutorial: Manipulação de Grafos em Larga Escala (Binários)

Este guia ensina a ler e manipular os formatos binários gerados pelo Java. O objetivo é dar ao time de Data Science o controle total sobre a memória e a velocidade de processamento.

---

In [1]:
import numpy as np
import pandas as pd
import networkx as nx
import os

PREFIX = 'facebook_union'
BASE_PATH = '../data/generated/bin/'

print(">>> Ambiente pronto para manipulação binária.")

>>> Ambiente pronto para manipulação binária.


## 1. A Lista de Arestas (Edge List)
**O que é:** Uma sequência simples de pares `(origem, destino)`. É a forma mais intuitiva de representar conexões.

**Estrutura no arquivo:**
1. `int32` (4 bytes): Ordem V
2. `int32` (4 bytes): Tamanho E
3. Sequência de E x 2 inteiros representando as arestas.

In [23]:
path = f"{BASE_PATH}{PREFIX}_edgelist.bin"
raw_data = np.fromfile(path, dtype='>i4') # Big-Endian

V = raw_data[0]
E = raw_data[1]
edges = raw_data[2:].reshape((E, 2))

print(f"Meta: V={V}, E={E}")
print(f"Primeiras 5 arestas:\n{edges[:5]}") 

# MANIPULAÇÃO: Converter para NetworkX instantaneamente
G_nx = nx.Graph()
G_nx.add_edges_from(edges)
print(f"Grafo NetworkX criado com {G_nx.number_of_nodes()} nós.")


Meta: V=4039, E=88234
Primeiras 5 arestas:
[[  0 347]
 [  0 346]
 [  0 345]
 [  0 344]
 [  0 343]]
Grafo NetworkX criado com 4039 nós.


## 2. Compressed Sparse Row (CSR)
**O que é:** A estrutura mais amada pela computação científica. Ela separa o 'onde começa' (Offsets) do 'quem é' (Edges).

**Estrutura:**
- `Offsets`: Um array onde o valor no índice `i` indica a posição inicial dos amigos do usuário `i` no array de conexões.
- `Edges`: Um linhão com todos os amigos de todo mundo grudados.

In [ ]:
path = f"{BASE_PATH}{PREFIX}_csr.bin"
data = np.fromfile(path, dtype='>i4')

v_count = data[0]
e_count = data[1]
offsets = data[2 : 2 + v_count + 1]
csr_data = data[2 + v_count + 1 :]

def get_neighbors(node_id):
    """Busca vizinhos em tempo O(1) - Velocidade máxima!"""
    start = offsets[node_id]
    end = offsets[node_id + 1]
    return csr_data[start:end]

# MANIPULAÇÃO: Calcular grau de um nó sem percorrer o grafo todo
target = 41 # Esse nó é o ID da pessoa, ou seja essa é a pessoa 41 da rede social
neighbors = get_neighbors(target)
print(f"Usuário {target} tem {len(neighbors) -1} amigos.") # Aqui é o numero de amigos do usuário 41, ou seja, o numero de vizinhos do nó 41, ou seja, o numero de conexões do nó 41. O -1 é para não contar o próprio nó como amigo.
print(f"Lista de amigos: {neighbors[0:-1]}...") # A lista impressa dos amigos do usuário 41: primeiro parametro é o inicio e o segundo é o fim, ou seja, o -1 é para não incluir o ultimo elemento que é o proprio nó.

Usuário 41 tem 23 amigos.
Lista de amigos: [343 337 326 312 310 243 230 226 214 151 144 140 137 116 115 111  93  44
  28  20  19  17  14]...


## 3. Matriz de Adjacência (Bitset)
**O que é:** Uma grade V x V onde 1 significa conectado e 0 desconectado. No nosso arquivo, cada conexão ocupa 1 bit.

**Atenção ao Parsing:**
O Java salva o BitSet de forma compacta. Precisamos expandir esses bits para uma matriz NumPy.

In [ ]:
path = f"{BASE_PATH}{PREFIX}_adjmatrix.bin"
v_header = np.fromfile(path, dtype='>i4', count=1)[0]
raw_bytes = np.fromfile(path, dtype=np.uint8, offset=4)

# Expansão: 1 byte -> 8 bits (0 ou 1)
bits = np.unpackbits(raw_bytes, bitorder='little')

# Ajuste de tamanho (padding)
total_size = v_header * v_header
if len(bits) < total_size:
    bits = np.pad(bits, (0, total_size - len(bits)))

adj_matrix = bits[:total_size].reshape((v_header, v_header))

print(f"Matriz expandida para memória: {adj_matrix.shape}")

# MANIPULAÇÃO: Ver se usuário A e B são amigos direto na matriz
a, b = 0, 1
is_friend = adj_matrix[a, b] == 1
print(f"Usuários {a} e {b} são amigos? {'Sim' if is_friend else 'Não'}")

Matriz expandida para memória: (4039, 4039)
Usuários 0 e 1 são amigos? Sim
Usuário 0 é amigo de [0 1 1 ... 0 0 0]? Não
Usuário 0 é amigo de [1 0 0 ... 0 0 0]? Não
Usuário 0 é amigo de [1 0 0 ... 0 0 0]? Não
Usuário 0 é amigo de [1 0 0 ... 0 0 0]? Não
Usuário 0 é amigo de [1 0 0 ... 0 0 0]? Não
Usuário 0 é amigo de [1 0 0 ... 0 0 0]? Não
Usuário 0 é amigo de [1 0 0 ... 0 0 0]? Não
Usuário 0 é amigo de [1 0 0 ... 0 0 0]? Não
Usuário 0 é amigo de [1 0 0 ... 0 0 0]? Não
Usuário 0 é amigo de [1 0 0 ... 0 0 0]? Não
Usuário 0 é amigo de [1 0 0 ... 0 0 0]? Não
Usuário 0 é amigo de [1 0 0 ... 0 0 0]? Não
Usuário 0 é amigo de [1 0 0 ... 0 0 0]? Não
Usuário 0 é amigo de [1 0 0 ... 0 0 0]? Não
Usuário 0 é amigo de [1 0 0 ... 0 0 0]? Não
Usuário 0 é amigo de [1 0 0 ... 0 0 0]? Não
Usuário 0 é amigo de [1 0 0 ... 0 0 0]? Não
Usuário 0 é amigo de [1 0 0 ... 0 0 0]? Não
Usuário 0 é amigo de [1 0 0 ... 0 0 0]? Não
Usuário 0 é amigo de [1 0 0 ... 0 0 0]? Não
Usuário 0 é amigo de [1 0 1 ... 0 0 0]? Não
U

## 4. Desafio de Performance: CSV vs Binário
Veja a diferença de tempo para ler os mesmos dados.

In [5]:
import time

start = time.time()
np.fromfile(f"{BASE_PATH}{PREFIX}_edgelist.bin", dtype='>i4')
print(f"Tempo Binário: {time.time() - start:.6f} segundos")

csv_path = '../data/generated/sheets/facebook_union_vertex_degrees.csv'
if os.path.exists(csv_path):
    start = time.time()
    pd.read_csv(csv_path)
    print(f"Tempo CSV (Pandas): {time.time() - start:.6f} segundos")

Tempo Binário: 0.003690 segundos
Tempo CSV (Pandas): 0.008894 segundos
